# 02 — Retrieval Evaluation

Đánh giá chất lượng retrieval sau `run.py retrieve`.

**Mục tiêu:**
- Chạy thử retrieval cho câu hỏi mẫu
- Kiểm tra silver recall
- So sánh BM25 vs Dense vs Hybrid

In [ ]:
import sys
sys.path.insert(0, '../src')

import json, pickle
import numpy as np
from vpl.settings import DATA_DIR, BM25_CORPUS_FILE, BM25_ID_MAP_FILE
from vpl.cache import RetrievalCache
from vpl.store.bm25 import tokenize

## 1. Test retrieval cho câu hỏi mẫu

In [ ]:
# Load retriever (cần đã chạy run.py index)
from vpl.store.vectors import get_collection
from vpl.search.hybrid import HybridRetriever
from vpl.search.reranker import load_reranker

collection, embed_model = get_collection(device='cpu')
reranker = load_reranker(device='cpu')
retriever = HybridRetriever(
    chroma_collection=collection,
    embedding_model=embed_model,
    reranker=reranker,
)
print('Retriever ready')

In [ ]:
test_queries = [
    'Doanh nghiệp nhỏ và vừa phải đáp ứng điều kiện nào để được hỗ trợ?',
    'Mức xử phạt vi phạm hành chính về thuế là bao nhiêu?',
    'Người lao động có quyền gì khi bị sa thải?',
    'Thủ tục đăng ký kinh doanh cho hộ kinh doanh?',
]

for query in test_queries:
    print(f'\nQuery: {query}')
    results = retriever.retrieve(query)
    for i, r in enumerate(results[:5], 1):
        m = r.metadata
        print(f'  {i}. [{r.score:.3f}] {m.get("doc_id")} {m.get("article_number")} — {m.get("doc_title","")[:50]}')

## 2. Silver Recall Check

In [ ]:
from vpl.evaluate import silver_recall

questions = json.loads((DATA_DIR / 'R2AIStage1DATA.json').read_text(encoding='utf-8'))
cache = RetrievalCache()
print(f'Cached: {len(cache.completed_ids)} / {len(questions)}')

if len(cache.completed_ids) > 100:
    cached_rows = []
    for q in questions:
        qid = int(q['id'])
        if qid in cache.completed_ids:
            try:
                chunks = cache.get(qid)
                cached_rows.append({'id': qid, 'question': q['question'], 'chunks': chunks})
            except Exception:
                pass
    metrics = silver_recall(questions, cached_rows)
    print('Silver Recall:')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
else:
    print('Cần chạy `python run.py retrieve` trước')

## 3. BM25 standalone test

In [ ]:
with BM25_CORPUS_FILE.open('rb') as f:
    bm25 = pickle.load(f)
bm25_ids = json.loads(BM25_ID_MAP_FILE.read_text())

query = 'điều kiện hỗ trợ vay vốn doanh nghiệp nhỏ và vừa'
tokens = tokenize(query)
scores = np.array(bm25.get_scores(tokens))
top5 = np.argsort(scores)[::-1][:5]

print(f'BM25 Top 5: "{query}"')
for idx in top5:
    print(f'  [{scores[idx]:.3f}] {bm25_ids[idx]}')